In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


BRONZE_SCHEMA = "football_data_platform.bronze"
SILVER_SCHEMA = "football_data_platform.silver"
SILVER_TABLE = f"{SILVER_SCHEMA}.players_statistics"

def clean_strings(df: DataFrame) -> DataFrame:
    """Trim string columns and convert empty strings to NULL."""

    for column, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column,
                F.when(
                    F.trim(F.col(f"`{column}`")) == "",
                    None
                ).otherwise(
                    F.trim(F.col(f"`{column}`"))
                )
            )

    return df


# ---------------------------------------------------------
# 1. Read Bronze table
# ---------------------------------------------------------

table_name = f"{BRONZE_SCHEMA}.api_players_statistics"

if not spark.catalog.tableExists(table_name):
    print(f"WARNING: Table does not exist: {table_name}")

df = spark.table(table_name)
df = clean_strings(df)

# ---------------------------------------------------------
# 2. Select and standardize columns
# ---------------------------------------------------------

df_silver = (
    df
    .select(
        F.col("player"),
        F.explode("statistics").alias("statistics"),
        F.col("_ingested_at").cast("timestamp").alias("_bronze_ingested_at"),
        F.col("_source_file").cast("string").alias("_bronze_source_file"),
        F.col("_source_system").cast("string").alias("_source_system")
    )
    .select(

        # Player
        F.col("player.id").cast("long").alias("player_id"),
        F.col("player.name").cast("string").alias("player_name"),
        F.col("player.firstname").cast("string").alias("player_firstname"),
        F.col("player.lastname").cast("string").alias("player_lastname"),
        F.col("player.age").cast("int").alias("player_age"),
        F.col("player.nationality").cast("string").alias("player_nationality"),
        F.col("player.height").cast("string").alias("player_height"),
        F.col("player.weight").cast("string").alias("player_weight"),
        F.col("player.injured").cast("boolean").alias("player_injured"),

        F.col("player.birth.date").cast("string").alias("player_birth_date"),
        F.col("player.birth.place").cast("string").alias("player_birth_place"),
        F.col("player.birth.country").cast("string").alias("player_birth_country"),

        # League
        F.col("statistics.league.id").cast("long").alias("league_id"),
        F.col("statistics.league.name").cast("string").alias("league_name"),
        F.col("statistics.league.country").cast("string").alias("league_country"),
        F.col("statistics.league.season").cast("int").alias("league_season"),

        # Team
        F.col("statistics.team.id").cast("long").alias("team_id"),
        F.col("statistics.team.name").cast("string").alias("team_name"),

        # Games
        F.col("statistics.games.appearences").cast("int").alias("games_appearances"),
        F.col("statistics.games.lineups").cast("int").alias("games_lineups"),
        F.col("statistics.games.minutes").cast("int").alias("games_minutes"),
        F.col("statistics.games.position").cast("string").alias("games_position"),
        F.col("statistics.games.rating").cast("double").alias("games_rating"),
        F.col("statistics.games.captain").cast("boolean").alias("games_captain"),
        F.col("statistics.games.number").cast("string").alias("games_number"),

        # Goals
        F.col("statistics.goals.total").cast("int").alias("goals_total"),
        F.col("statistics.goals.assists").cast("int").alias("goals_assists"),
        F.col("statistics.goals.conceded").cast("int").alias("goals_conceded"),
        F.col("statistics.goals.saves").cast("int").alias("goals_saves"),

        # Shots
        F.col("statistics.shots.total").cast("int").alias("shots_total"),
        F.col("statistics.shots.on").cast("int").alias("shots_on"),

        # Passes
        F.col("statistics.passes.total").cast("int").alias("passes_total"),
        F.col("statistics.passes.key").cast("int").alias("passes_key"),
        F.col("statistics.passes.accuracy").cast("int").alias("passes_accuracy"),

        # Tackles
        F.col("statistics.tackles.total").cast("int").alias("tackles_total"),
        F.col("statistics.tackles.blocks").cast("int").alias("tackles_blocks"),
        F.col("statistics.tackles.interceptions").cast("int").alias("tackles_interceptions"),

        # Duels
        F.col("statistics.duels.total").cast("int").alias("duels_total"),
        F.col("statistics.duels.won").cast("int").alias("duels_won"),

        # Dribbles
        F.col("statistics.dribbles.attempts").cast("int").alias("dribbles_attempts"),
        F.col("statistics.dribbles.success").cast("int").alias("dribbles_success"),
        F.col("statistics.dribbles.past").cast("string").alias("dribbles_past"),

        # Fouls
        F.col("statistics.fouls.committed").cast("int").alias("fouls_committed"),
        F.col("statistics.fouls.drawn").cast("int").alias("fouls_drawn"),

        # Cards
        F.col("statistics.cards.yellow").cast("int").alias("cards_yellow"),
        F.col("statistics.cards.yellowred").cast("int").alias("cards_yellowred"),
        F.col("statistics.cards.red").cast("int").alias("cards_red"),

        # Substitutes
        F.col("statistics.substitutes.bench").cast("int").alias("substitutes_bench"),
        F.col("statistics.substitutes.in").cast("int").alias("substitutes_in"),
        F.col("statistics.substitutes.out").cast("int").alias("substitutes_out"),

        # Penalties
        F.col("statistics.penalty.commited").cast("string").alias("penalty_committed"),
        F.col("statistics.penalty.missed").cast("int").alias("penalty_missed"),
        F.col("statistics.penalty.saved").cast("int").alias("penalty_saved"),
        F.col("statistics.penalty.scored").cast("int").alias("penalty_scored"),
        F.col("statistics.penalty.won").cast("string").alias("penalty_won"),

        # Bronze metadata
        F.col("_bronze_ingested_at"),
        F.col("_bronze_source_file"),
        F.col("_source_system")
    )
)

# ---------------------------------------------------------
# 3. Remove duplicates
# ---------------------------------------------------------

duplicate_columns = [
    "player_id",
    "team_id",
    "league_id",
    "league_season",
]

duplicate_count = (
    df_silver
    .groupBy(duplicate_columns)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate groups found: {duplicate_count}")

if duplicate_count > 0:
    df_silver = df_silver.dropDuplicates(duplicate_columns)
    print("Duplicates removed.")
else:
    print("No duplicates found. No changes made.")


# ---------------------------------------------------------
# 4. Add Silver metadata
# ---------------------------------------------------------

df_silver = df_silver.withColumn(
    "_silver_processed_at",
    F.current_timestamp()
)


# ---------------------------------------------------------
# 5. Basic validation
# ---------------------------------------------------------

# Null league IDs
null_league_ids = df_silver.filter(
    F.col("league_id").isNull()
).count()

if null_league_ids > 0:
    print(f"WARNING: NULL league_id: {null_league_ids}")


# Null player IDs
null_player_ids = df_silver.filter(
    F.col("player_id").isNull()
).count()

if null_player_ids > 0:
    print(f"WARNING: NULL player_id: {null_player_ids}")
    
print(f"Silver rows: {df_silver.count()}")

# ---------------------------------------------------------
# 6. Write Silver
# ---------------------------------------------------------

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Created {SILVER_TABLE}")